# Four-way State Estimation Uncertainty Quantification Comparison

## Goal

This tutorial compares uncertainty outputs from four state-estimation UQ paths:

1. analytical UQ with iterative-linear state estimation (ILSE);
2. analytical UQ with Newton–Raphson state estimation (NRSE);
3. Monte Carlo UQ with ILSE; and
4. Monte Carlo UQ with NRSE.

All four paths use the same observable network and sensor data. The two analytical paths calculate local first-order output standard deviations directly, while the two Monte Carlo paths estimate output standard deviations from the same 5,000 reproducible noisy measurement scenarios.

See the [ILSE formulation](../algorithms/se-algorithms.md#state-estimate-output-uncertainty) and [NRSE formulation](../algorithms/se-algorithms.md#newton-raphson-output-uncertainty) for the analytical details.


## Setup

The notebook uses the editable Python package and native library from the current Power Grid Model checkout. NumPy performs checks, while pandas presents compact comparison tables.

In [1]:
import sys
from time import perf_counter

import numpy as np
import pandas as pd

import power_grid_model
from power_grid_model import (
    AttributeType,
    CalculationMethod,
    CalculationType,
    ComponentType,
    DatasetType,
    LoadGenType,
    MeasuredTerminalType,
    PowerGridModel,
    initialize_array,
)
from power_grid_model._core.power_grid_model_c.get_pgm_dll_path import get_pgm_dll_path
from power_grid_model.utils import create_state_estimation_monte_carlo_updates
from power_grid_model.validation import assert_valid_input_data

print("Python:", sys.executable)
print("Power Grid Model package:", power_grid_model.__file__)
print("Native library:", get_pgm_dll_path())

Python: /home/jhe/pgm_dev/power-grid-model/.venv/bin/python
Power Grid Model package: /home/jhe/pgm_dev/power-grid-model/src/power_grid_model/__init__.py
Native library: /home/jhe/pgm_dev/power-grid-model/build/bin/libpower_grid_model_c.so


## Steps

### 1. Build the network and measurements

The example uses a two-node network. A voltage-magnitude sensor observes the source node, a power-flow sensor observes the line, and a load power sensor provides the remaining measurement information.

```text
source 41 -- node 11 -- line 21 -- node 12 -- load 31
              |             |               |
       voltage sensor 71    |        power sensor 62
                       line power sensor 61
```

All values use SI units. The line measurement includes the approximate resistive loss, so it is consistent with the load measurement. Sensor sigma attributes describe input measurement errors; they are distinct from the output sigma attributes calculated below. No voltage-angle measurement is present, so reported angles use phase A of the slack bus as their reference.

In [2]:
node = initialize_array(DatasetType.input, ComponentType.node, 2)
node[AttributeType.id] = [11, 12]
node[AttributeType.u_rated] = [10.5e3, 10.5e3]

line = initialize_array(DatasetType.input, ComponentType.line, 1)
line[AttributeType.id] = [21]
line[AttributeType.from_node] = [11]
line[AttributeType.to_node] = [12]
line[AttributeType.from_status] = [1]
line[AttributeType.to_status] = [1]
line[AttributeType.r1] = [0.1]
line[AttributeType.x1] = [0.0]
line[AttributeType.c1] = [0.0]
line[AttributeType.tan1] = [0.0]
line[AttributeType.i_n] = [510.0]

source = initialize_array(DatasetType.input, ComponentType.source, 1)
source[AttributeType.id] = [41]
source[AttributeType.node] = [11]
source[AttributeType.status] = [1]
source[AttributeType.u_ref] = [1.0]

sym_load = initialize_array(DatasetType.input, ComponentType.sym_load, 1)
sym_load[AttributeType.id] = [31]
sym_load[AttributeType.node] = [12]
sym_load[AttributeType.status] = [1]
sym_load[AttributeType.type] = [LoadGenType.const_power]

sym_voltage_sensor = initialize_array(DatasetType.input, ComponentType.sym_voltage_sensor, 1)
sym_voltage_sensor[AttributeType.id] = [71]
sym_voltage_sensor[AttributeType.measured_object] = [11]
sym_voltage_sensor[AttributeType.u_measured] = [10.5e3]
sym_voltage_sensor[AttributeType.u_sigma] = [105.0]

sym_power_sensor = initialize_array(DatasetType.input, ComponentType.sym_power_sensor, 2)
sym_power_sensor[AttributeType.id] = [61, 62]
sym_power_sensor[AttributeType.measured_object] = [21, 31]
sym_power_sensor[AttributeType.measured_terminal_type] = [
    MeasuredTerminalType.branch_from,
    MeasuredTerminalType.load,
]
sym_power_sensor[AttributeType.p_measured] = [1.001e6, 1.0e6]
sym_power_sensor[AttributeType.q_measured] = [0.0, 0.0]
sym_power_sensor[AttributeType.power_sigma] = [1.0e3, 1.0e3]

input_data = {
    ComponentType.node: node,
    ComponentType.line: line,
    ComponentType.source: source,
    ComponentType.sym_load: sym_load,
    ComponentType.sym_voltage_sensor: sym_voltage_sensor,
    ComponentType.sym_power_sensor: sym_power_sensor,
}

assert_valid_input_data(input_data, calculation_type=CalculationType.state_estimation)

### 2. Run analytical UQ with ILSE and NRSE

`calculate_uncertainty` defaults to `False`. Sigma fields remain in the output schema but contain `NaN` until UQ is requested. The same model and input data can be used with either supported method.

The checks below also confirm that enabling analytical UQ does not change either method's point estimate.


In [3]:
model = PowerGridModel(input_data, system_frequency=50.0)
methods = {
    "ILSE": CalculationMethod.iterative_linear,
    "NRSE": CalculationMethod.newton_raphson,
}

results_without_uq = {}
results = {}
mean_fields = {
    ComponentType.node: ["u_pu", "u", "u_angle", "p", "q"],
    ComponentType.line: ["p_from", "q_from", "i_from", "p_to", "q_to", "i_to"],
}
sigma_fields = {
    ComponentType.node: ["u_pu_sigma", "u_sigma", "u_angle_sigma", "p_sigma", "q_sigma"],
    ComponentType.line: [
        "p_from_sigma",
        "q_from_sigma",
        "i_from_sigma",
        "p_to_sigma",
        "q_to_sigma",
        "i_to_sigma",
    ],
}

for method_name, calculation_method in methods.items():
    results_without_uq[method_name] = model.calculate_state_estimation(
        calculation_method=calculation_method,
    )
    results[method_name] = model.calculate_state_estimation(
        calculation_method=calculation_method,
        calculate_uncertainty=True,
    )

    for component, fields in sigma_fields.items():
        for field in fields:
            sigma_without_uq = results_without_uq[method_name][component][field]
            sigma = results[method_name][component][field]
            expected_mask = np.ones_like(sigma, dtype=bool)
            np.testing.assert_array_equal(np.isnan(sigma_without_uq), expected_mask)
            np.testing.assert_array_equal(np.isfinite(sigma), expected_mask)
            np.testing.assert_array_equal(sigma >= 0.0, expected_mask)

    for component, fields in mean_fields.items():
        for field in fields:
            np.testing.assert_allclose(
                results[method_name][component][field],
                results_without_uq[method_name][component][field],
            )

### 3. Compare node uncertainty

Each sigma has the same physical unit and shape as its corresponding output. Angle sigmas are in radians. Because there is no physical angle measurement, phase A of the slack bus is the reporting reference and its `u_angle_sigma` is zero for both methods.

In [4]:
node_columns = [
    "id",
    "u_pu",
    "u_pu_sigma",
    "u",
    "u_sigma",
    "u_angle",
    "u_angle_sigma",
    "p",
    "p_sigma",
    "q",
    "q_sigma",
]
node_results = {
    method_name: pd.DataFrame(method_result[ComponentType.node])[node_columns]
    for method_name, method_result in results.items()
}
for method_name, node_result in node_results.items():
    np.testing.assert_equal(node_result.loc[0, "u_angle_sigma"], 0.0)

pd.concat(
    [node_result.assign(method=method_name) for method_name, node_result in node_results.items()],
    ignore_index=True,
)[["method", *node_columns]]

,method,id,u_pu,u_pu_sigma,u,u_sigma,u_angle,u_angle_sigma,p,p_sigma,q,q_sigma
0,ILSE,11,1.000000,0.007071,10499.999990,74.246212,0.0,0.000000e+00,1.000954e+06,7095.454992,0.0,7095.454992
1,ILSE,12,0.999092,0.007071,10490.467092,74.246212,0.0,6.441635e-06,-1.000046e+06,7095.391072,0.0,7095.454992
2,NRSE,11,0.999983,0.009998,10499.825772,104.982628,0.0,0.000000e+00,1.000954e+06,500.536652,0.0,500.000000
3,NRSE,12,0.999075,0.010007,10490.292714,105.078031,0.0,4.539419e-07,-1.000046e+06,499.628353,0.0,500.000000


### 4. Compare branch uncertainty

For a two-terminal branch, current-magnitude and active/reactive-power sigmas are available at both sides. There is no `s_sigma`; active and reactive uncertainty are reported separately.

In [5]:
line_columns = [
    "id",
    "p_from",
    "p_from_sigma",
    "q_from",
    "q_from_sigma",
    "i_from",
    "i_from_sigma",
    "p_to",
    "p_to_sigma",
    "q_to",
    "q_to_sigma",
    "i_to",
    "i_to_sigma",
]
line_results = {
    method_name: pd.DataFrame(method_result[ComponentType.line])[line_columns]
    for method_name, method_result in results.items()
}

pd.concat(
    [line_result.assign(method=method_name) for method_name, line_result in line_results.items()],
    ignore_index=True,
)[["method", *line_columns]]

,method,id,p_from,p_from_sigma,q_from,q_from_sigma,i_from,i_from_sigma,p_to,p_to_sigma,q_to,q_to_sigma,i_to,i_to_sigma
0,ILSE,21,1.000954e+06,7095.454992,0.0,7095.454992,55.038216,0.027493,-1.000046e+06,7095.391072,-0.0,7095.454992,55.038216,0.027493
1,NRSE,21,1.000954e+06,500.536643,0.0,499.999995,55.039134,0.551496,-1.000046e+06,499.628337,-0.0,499.999995,55.039134,0.551496


### 5. Compare the method-specific sigmas

ILSE and NRSE share the API, not the covariance model:

- ILSE uses PGM's adopted proper-complex voltage-error model. Real marginals contain its factor of $1/2$.
- NRSE uses a real polar state $[\theta, v]$ and a frozen Gauss–Newton covariance rebuilt at the returned state. Ordinary real propagation applies without the proper-complex factor.

The comparison below therefore treats both results as valid method-specific answers rather than expecting them to match.

In [6]:
pd.DataFrame(
    {
        "method": list(methods),
        "slack u_sigma [V]": [node_results[name].loc[0, "u_sigma"] for name in methods],
        "remote u_angle_sigma [rad]": [node_results[name].loc[1, "u_angle_sigma"] for name in methods],
        "line i_from_sigma [A]": [line_results[name].loc[0, "i_from_sigma"] for name in methods],
        "line p_from_sigma [W]": [line_results[name].loc[0, "p_from_sigma"] for name in methods],
        "line q_from_sigma [var]": [line_results[name].loc[0, "q_from_sigma"] for name in methods],
    }
)

,method,slack u_sigma [V],remote u_angle_sigma [rad],line i_from_sigma [A],line p_from_sigma [W],line q_from_sigma [var]
0,ILSE,74.246212,6.441635e-06,0.027493,7095.454992,7095.454992
1,NRSE,104.982628,4.539419e-07,0.551496,500.536643,499.999995


### 6. Reproduce the simple ILSE current and power checks

For ILSE, the voltage sensor directly supplies a magnitude sigma. A symmetric `power_sigma` describes a circular complex error, so each power sensor contributes $\sigma_P=\sigma_Q=\sigma_S/\sqrt{2}$. In this series network, the two independent power sensors constrain the same current, adding another factor of $1/\sqrt{2}$ when their information is combined.

Reconstructing $S=UI^*$ also propagates absolute-voltage uncertainty. These simplified closed-form checks are specific to ILSE and this two-node example; they are not reused as NRSE expectations.

In [7]:
il_node_result = node_results["ILSE"]
il_line_result = line_results["ILSE"]
input_voltage_sigma = float(sym_voltage_sensor[AttributeType.u_sigma][0])
input_apparent_power_sigma = float(sym_power_sensor[AttributeType.power_sigma][0])
input_component_power_sigma = input_apparent_power_sigma / np.sqrt(2.0)
combined_component_power_sigma = input_component_power_sigma / np.sqrt(sym_power_sensor.size)

measured_voltage = float(sym_voltage_sensor[AttributeType.u_measured][0])
expected_current_sigma = combined_component_power_sigma / (np.sqrt(3.0) * measured_voltage)
reported_current_sigma = float(il_line_result.loc[0, "i_from_sigma"])
np.testing.assert_allclose(reported_current_sigma, expected_current_sigma)

p_from = abs(float(il_line_result.loc[0, "p_from"]))
u_from = float(il_node_result.loc[0, "u"])
i_from = float(il_line_result.loc[0, "i_from"])
voltage_power_contribution = p_from * float(il_node_result.loc[0, "u_sigma"]) / u_from
current_power_contribution = p_from * reported_current_sigma / i_from
expected_power_sigma = np.hypot(voltage_power_contribution, current_power_contribution)

np.testing.assert_allclose(il_line_result.loc[0, "p_from_sigma"], expected_power_sigma)
np.testing.assert_allclose(il_line_result.loc[0, "q_from_sigma"], expected_power_sigma)

pd.DataFrame(
    {
        "quantity": ["voltage magnitude", "current magnitude", "active power", "reactive power"],
        "unit": ["V", "A", "W", "var"],
        "predicted ILSE sigma": [
            input_voltage_sigma / np.sqrt(2.0),
            expected_current_sigma,
            expected_power_sigma,
            expected_power_sigma,
        ],
        "reported ILSE sigma": [
            il_node_result.loc[0, "u_sigma"],
            reported_current_sigma,
            il_line_result.loc[0, "p_from_sigma"],
            il_line_result.loc[0, "q_from_sigma"],
        ],
    }
)

,quantity,unit,predicted ILSE sigma,reported ILSE sigma
0,voltage magnitude,V,74.246212,74.246212
1,current magnitude,A,0.027493,0.027493
2,active power,W,7095.454992,7095.454992
3,reactive power,var,7095.454992,7095.454992


### 7. Run Monte Carlo UQ with ILSE and NRSE

`create_state_estimation_monte_carlo_updates` turns the sensor sigmas into one reproducible dense batch. The same 5,000 noisy measurement scenarios are passed to ILSE and NRSE, so their empirical output standard deviations and execution times are directly comparable.

This is a **raw-sensor Monte Carlo model**: voltage/current magnitude and angle are sampled in their public polar coordinates, while `power_sigma` becomes independent P and Q errors with sigma $\sigma_S/\sqrt{2}$. It is a direct numerical cross-check for NRSE on this example. It is deliberately not an exact oracle for ILSE, whose analytical UQ adopts an internal proper-complex effective voltage-error model with a factor of $1/2$. Differences between analytical ILSE and Monte Carlo ILSE therefore expose a modeling distinction rather than a failed implementation check.

Some noisy NRSE scenarios in this intentionally simple zero-reactance case do not converge. The benchmark records and excludes those scenarios instead of silently treating their last iterate as a sample.


In [8]:
monte_carlo_sample_count = 5_000
monte_carlo_updates = create_state_estimation_monte_carlo_updates(
    input_data,
    monte_carlo_sample_count,
    seed=2026,
)

monte_carlo_results = {}
monte_carlo_valid_samples = {}
benchmark_rows = []

for method_name, calculation_method in methods.items():
    benchmark_model = PowerGridModel(input_data, system_frequency=50.0)

    analytical_start = perf_counter()
    benchmark_model.calculate_state_estimation(
        calculation_method=calculation_method,
        calculate_uncertainty=True,
        error_tolerance=1e-10,
        max_iterations=50,
    )
    analytical_seconds = perf_counter() - analytical_start

    monte_carlo_start = perf_counter()
    sample_result = benchmark_model.calculate_state_estimation(
        calculation_method=calculation_method,
        update_data=monte_carlo_updates,
        error_tolerance=1e-8,
        max_iterations=50,
        continue_on_batch_error=True,
    )
    monte_carlo_seconds = perf_counter() - monte_carlo_start

    failed_scenarios = [] if benchmark_model.batch_error is None else benchmark_model.batch_error.failed_scenarios
    valid_samples = np.ones(monte_carlo_sample_count, dtype=bool)
    valid_samples[failed_scenarios] = False
    np.testing.assert_array_less(0.95 * monte_carlo_sample_count, np.count_nonzero(valid_samples))

    monte_carlo_results[method_name] = sample_result
    monte_carlo_valid_samples[method_name] = valid_samples
    benchmark_rows.append(
        {
            "method": method_name,
            "analytical UQ [ms]": 1e3 * analytical_seconds,
            "Monte Carlo batch [ms]": 1e3 * monte_carlo_seconds,
            "attempted samples": monte_carlo_sample_count,
            "converged samples": int(np.count_nonzero(valid_samples)),
            "MC [µs / attempted sample]": 1e6 * monte_carlo_seconds / monte_carlo_sample_count,
        }
    )

pd.DataFrame(benchmark_rows).round(3)

,method,analytical UQ [ms],Monte Carlo batch [ms],attempted samples,converged samples,MC [µs / attempted sample]
0,ILSE,0.175,18.867,5000,5000,3.773
1,NRSE,0.150,37.153,5000,4934,7.431


### 8. Compare all four UQ outputs

The four numerical columns below are the requested comparison. Each entry is an output standard deviation in the unit shown. The table includes every non-redundant node and line UQ output for this network; `u_pu_sigma` is omitted because it is the per-unit normalization of `u_sigma`.

The **Original sensor sigma** column reports the configured input field and sensor ID associated with that physical quantity. A dash means there is no corresponding sensor. In this one-branch network, sensor 61 constrains node 11 injection and line-from power, while load sensor 62 constrains node 12 injection and line-to power.

The configured `power_sigma = 1000 VA` describes a circular complex-power error. Therefore, the Monte Carlo generator samples each independent P and Q channel with $1000/\sqrt{2}=707.107$ W or var. This conversion is important when comparing the original input value with the output sigmas.

Monte Carlo values use the sample standard deviation (`ddof=1`) over converged scenarios. A fixed-seed check verifies selected NRSE Monte Carlo values against analytical NRSE within 8%. No equality is asserted for analytical versus Monte Carlo ILSE because their effective voltage-noise models differ, as described above.


In [9]:
comparison_fields = [
    ("node 11 voltage magnitude", ComponentType.node, "u", "u_sigma", 0, "V", "105 V (u_sigma, sensor 71)"),
    ("node 11 voltage angle", ComponentType.node, "u_angle", "u_angle_sigma", 0, "rad", "—"),
    ("node 11 active injection", ComponentType.node, "p", "p_sigma", 0, "W", "1000 VA (power_sigma, sensor 61)"),
    ("node 11 reactive injection", ComponentType.node, "q", "q_sigma", 0, "var", "1000 VA (power_sigma, sensor 61)"),
    ("node 12 voltage magnitude", ComponentType.node, "u", "u_sigma", 1, "V", "—"),
    ("node 12 voltage angle", ComponentType.node, "u_angle", "u_angle_sigma", 1, "rad", "—"),
    ("node 12 active injection", ComponentType.node, "p", "p_sigma", 1, "W", "1000 VA (power_sigma, sensor 62)"),
    ("node 12 reactive injection", ComponentType.node, "q", "q_sigma", 1, "var", "1000 VA (power_sigma, sensor 62)"),
    ("line 21 from current", ComponentType.line, "i_from", "i_from_sigma", 0, "A", "—"),
    (
        "line 21 from active power",
        ComponentType.line,
        "p_from",
        "p_from_sigma",
        0,
        "W",
        "1000 VA (power_sigma, sensor 61)",
    ),
    (
        "line 21 from reactive power",
        ComponentType.line,
        "q_from",
        "q_from_sigma",
        0,
        "var",
        "1000 VA (power_sigma, sensor 61)",
    ),
    ("line 21 to current", ComponentType.line, "i_to", "i_to_sigma", 0, "A", "—"),
    ("line 21 to active power", ComponentType.line, "p_to", "p_to_sigma", 0, "W", "1000 VA (power_sigma, sensor 62)"),
    (
        "line 21 to reactive power",
        ComponentType.line,
        "q_to",
        "q_to_sigma",
        0,
        "var",
        "1000 VA (power_sigma, sensor 62)",
    ),
]
comparison_rows = []

for label, component, value_field, sigma_field, component_index, unit, sensor_sigma in comparison_fields:
    row = {"output": label, "unit": unit, "Original sensor sigma": sensor_sigma}
    for method_name in methods:
        row[f"Analytical UQ — {method_name}"] = float(results[method_name][component][sigma_field][component_index])
        valid_samples = monte_carlo_valid_samples[method_name]
        sampled_values = monte_carlo_results[method_name][component][value_field][valid_samples, component_index]
        row[f"Monte Carlo UQ — {method_name}"] = float(np.std(sampled_values, ddof=1))
    comparison_rows.append(row)

numeric_columns = [
    "Analytical UQ — ILSE",
    "Analytical UQ — NRSE",
    "Monte Carlo UQ — ILSE",
    "Monte Carlo UQ — NRSE",
]
four_method_comparison = pd.DataFrame(comparison_rows)[["output", "unit", "Original sensor sigma", *numeric_columns]]
np.testing.assert_array_equal(
    np.isfinite(four_method_comparison[numeric_columns]),
    np.ones((len(comparison_fields), len(numeric_columns)), dtype=np.bool_),
)

nr_validation_outputs = [
    "node 11 voltage magnitude",
    "node 12 voltage angle",
    "line 21 from current",
    "line 21 from active power",
]
nr_validation_rows = four_method_comparison.set_index("output").loc[nr_validation_outputs]
nr_ratios = (nr_validation_rows["Monte Carlo UQ — NRSE"] / nr_validation_rows["Analytical UQ — NRSE"]).to_numpy()
np.testing.assert_allclose(nr_ratios, 1.0, rtol=0.08)

four_method_comparison.style.format({column: "{:.6g}" for column in numeric_columns})

,output,unit,Original sensor sigma,Analytical UQ — ILSE,Analytical UQ — NRSE,Monte Carlo UQ — ILSE,Monte Carlo UQ — NRSE
0,node 11 voltage magnitude,V,"105 V (u_sigma, sensor 71)",74.2462,104.983,105.444,101.516
1,node 11 voltage angle,rad,—,0,0,0,0
2,node 11 active injection,W,"1000 VA (power_sigma, sensor 61)",7095.45,500.537,498.91,497.836
3,node 11 reactive injection,var,"1000 VA (power_sigma, sensor 61)",7095.45,500,503.193,503.702
4,node 12 voltage magnitude,V,—,74.2462,105.078,105.54,101.608
5,node 12 voltage angle,rad,—,6.44164e-06,4.53942e-07,4.57095e-07,4.57299e-07
6,node 12 active injection,W,"1000 VA (power_sigma, sensor 62)",7095.39,499.628,497.847,496.756
7,node 12 reactive injection,var,"1000 VA (power_sigma, sensor 62)",7095.45,500,503.193,503.702
8,line 21 from current,A,—,0.0274929,0.551496,0.554642,0.532985
9,line 21 from active power,W,"1000 VA (power_sigma, sensor 61)",7095.45,500.537,498.91,497.836


### 9. Why analytical ILSE differs from the other three paths

The discrepancy is expected because analytical ILSE propagates a different uncertainty model from the raw-sensor Monte Carlo experiment. It is conditional on ILSE's **final frozen measurement transformation** and adopts a **proper (circular) complex-voltage error**. Monte Carlo ILSE instead perturbs the public sensor coordinates and reruns the complete iterative transformation for every sample. Analytical NRSE uses a real polar covariance, so it is much closer to that raw polar Monte Carlo model.

The numbers in this example make both ILSE assumptions visible:

1. **The proper-complex factor changes voltage uncertainty.** The source voltage sensor has `u_sigma = 105 V`. Analytical ILSE divides a proper complex variance equally between its real and imaginary marginals, giving

   $$
   \sigma_{|U|,\mathrm{IL}}=\frac{105}{\sqrt{2}}=74.246\ \mathrm{V}.
   $$

   NRSE and both raw-sensor Monte Carlo paths perturb voltage magnitude directly with 105 V, so their voltage-magnitude sigmas remain near 105 V.

2. **Analytical ILSE freezes the power-to-current transformation.** Each configured `power_sigma = 1000 VA` becomes a P- or Q-channel sigma of $1000/\sqrt{2}=707.107$ W or var. The two independent power measurements constrain the same series flow, reducing the combined component sigma to about 500 W. In the frozen ILSE model this gives

   $$
   \sigma_{|I|,\mathrm{IL}}\approx\frac{500}{\sqrt{3}\,10500}=0.02749\ \mathrm{A}.
   $$

   End-to-end Monte Carlo recomputes the power-to-current conversion after perturbing the voltage sensor. The nominal current is about 55 A, so a 1% voltage perturbation alone produces roughly $55\times0.01=0.55$ A of current variation. That is why both Monte Carlo columns and analytical NRSE are near 0.53–0.55 A rather than 0.0275 A.

3. **The same frozen complex model inflates ILSE power and angle sigmas.** For line active power, the analytical ILSE voltage contribution is approximately

   $$
   |P|\frac{\sigma_{|U|,\mathrm{IL}}}{|U|}
   \approx 1.001\times10^6\frac{74.246}{10500}
   \approx 7079\ \mathrm{W}.
   $$

   Combining it with the roughly 500 W current contribution gives about 7095 W, matching the table. In the complete Monte Carlo rerun, voltage and current move together so that their product remains constrained by the power sensors; this coupling leaves an output spread near 500 W. The proper-complex ILSE assumption also supplies an unmeasured quadrature-voltage error, which explains its much larger remote-angle sigma.

Therefore, the table is not comparing four estimators under one identical stochastic model. It compares two analytical covariance conventions with two raw-sensor simulations. A Monte Carlo validation specifically targeting analytical ILSE would need to sample ILSE's frozen internal complex measurement model rather than perturbing the public polar measurements and rerunning every transformation.


## Checks

### Measurement-error scaling

Multiplying every measurement sigma by two preserves all relative weights. For both methods, the point estimate stays unchanged and every propagated output sigma doubles.

In [10]:
scaled_input_data = {component: values.copy() for component, values in input_data.items()}
scaled_input_data[ComponentType.sym_voltage_sensor][AttributeType.u_sigma] *= 2.0
scaled_input_data[ComponentType.sym_power_sensor][AttributeType.power_sigma] *= 2.0
scaled_model = PowerGridModel(scaled_input_data, system_frequency=50.0)
scaled_results = {}

for method_name, calculation_method in methods.items():
    scaled_results[method_name] = scaled_model.calculate_state_estimation(
        calculation_method=calculation_method,
        calculate_uncertainty=True,
    )
    for component, fields in mean_fields.items():
        for field in fields:
            np.testing.assert_allclose(
                scaled_results[method_name][component][field],
                results[method_name][component][field],
            )
    for component, fields in sigma_fields.items():
        for field in fields:
            np.testing.assert_allclose(
                scaled_results[method_name][component][field],
                2.0 * results[method_name][component][field],
            )

pd.concat(
    [
        pd.DataFrame(
            {
                "method": method_name,
                "node_id": results[method_name][ComponentType.node]["id"],
                "u_sigma [V]": results[method_name][ComponentType.node]["u_sigma"],
                "u_sigma after 2x input sigma [V]": scaled_results[method_name][ComponentType.node]["u_sigma"],
            }
        )
        for method_name in methods
    ],
    ignore_index=True,
)

,method,node_id,u_sigma [V],u_sigma after 2x input sigma [V]
0,ILSE,11,74.246212,148.492424
1,ILSE,12,74.246212,148.492424
2,NRSE,11,104.982628,209.965257
3,NRSE,12,105.078031,210.156063


## Next Steps

The four-way table compares output spread, but the four paths do not all encode the same uncertainty assumptions.

- Analytical ILSE describes the final fixed iterative-linear model and adopts a proper-complex effective error.
- Analytical NRSE uses the augmented Gauss–Newton matrix rebuilt at its final returned state. Without a physical angle measurement, deterministic virtual-angle contributions are removed before covariance is projected to the reported slack phase-A reference.
- Monte Carlo ILSE and Monte Carlo NRSE propagate the same raw public-coordinate sensor samples through their respective estimators.
- Both analytical methods assume independent processed measurement channels and first-order output propagation; the Monte Carlo paths include nonlinear estimator response but still assume the sampled sensor errors are independent Gaussian variables.
- Current-magnitude sigma is unavailable at exactly zero current. Ideal-link flow sigmas and individual injection sigmas inside a collapsed ideal-link supernode are also unavailable.
- A case requiring numerical LU pivot perturbation raises `SparseMatrixError` during analytical UQ.

Try replacing the network with an asymmetric three-phase model, adding a physical angle measurement, increasing the Monte Carlo sample count, or changing individual sensor sigmas to see how all four UQ paths respond.
